# ASTE Tools Synthetic Binning Demo

This notebook demonstrates the package API with synthetic arrays, so it does not require scratch ASTE output.

In [3]:
from pathlib import Path
import sys

CRIOS_BACKUPS_DIR = Path("/home/mmurakami/crios_backups")
ASTE_TOOLS_DIR = CRIOS_BACKUPS_DIR / "aste_tools"
ASTE_DIR = "/scratch/atnguyen/aste_270x450x180/OFFICIAL_ASTE_R1_Sep2019/"

if str(ASTE_TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(ASTE_TOOLS_DIR))

ASTE_TOOLS_DIR, ASTE_DIR

(PosixPath('/home/mmurakami/crios_backups/aste_tools'),
 '/scratch/atnguyen/aste_270x450x180/OFFICIAL_ASTE_R1_Sep2019/')

In [4]:
import numpy as np

from aste_tools.binning import bin_array, create_ts_mesh
from aste_tools.grid import ASTEGridSpec, compact_to_tracer, compact_uv_to_tracer, tracer_to_compact
from aste_tools.masks import masked_area_mean
from aste_tools.timing import monthly_file_steps

In [5]:
grid = ASTEGridSpec(nx=4, ncut1=5, ncut2=3, nz=2, name="tiny-demo")
compact = np.arange(np.prod(grid.compact_shape_3d), dtype=float).reshape(grid.compact_shape_3d)

tracer = compact_to_tracer(compact, grid)
roundtrip = tracer_to_compact(tracer, grid)

compact.shape, tracer.shape, np.nanmax(np.abs(compact - roundtrip))

((2, 17, 4), (2, 12, 8), 0.0)

In [6]:
u_plot, v_plot = compact_uv_to_tracer(compact, compact + 1000, grid)
u_plot.shape, v_plot.shape

((2, 13, 9), (2, 13, 9))

In [7]:
salt_edges = np.array([33.0, 34.0, 35.0, 36.0])
temp_edges = np.array([-2.0, 0.0, 2.0, 4.0])
salinity = np.array([[[33.5, 34.5], [35.5, np.nan]]])
theta = np.array([[[-1.0, 1.0], [3.0, 10.0]]])
volume = np.array([[[2.0, 3.0], [4.0, 5.0]]])

mesh = create_ts_mesh(
    volume,
    bin_array(salinity, salt_edges),
    bin_array(theta, temp_edges),
    n_salinity=3,
    n_theta=3,
)
mesh, np.nansum(mesh)

(array([[[2., 0., 0.],
         [0., 3., 0.],
         [0., 0., 4.]]]),
 9.0)

In [8]:
mean, area = masked_area_mean(compact[0], np.ones(grid.compact_shape_2d), np.ones(grid.compact_shape_2d))
steps = monthly_file_steps(delta_t=3600, start_year=1979, end_year=1980)
mean, area, steps[:3]

(33.5, 68.0, array([ 744, 1416, 2160]))